# MLP Semi-Infinite-Domain Hyperparameter Optimization

Optuna searches MLP depth, width, activation, and learning rate for the semi-infinite manufactured problem.

In [1]:
import os
import sys
from datetime import datetime
from importlib import reload

current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

import joblib
import optuna
import pandas as pd
import torch
import torch.nn as nn
import pinns_semi_infinite
import semi_infinite
from pinns_semi_infinite import run_experiment_semi_inf, set_seed

reload(semi_infinite)
reload(pinns_semi_infinite)
torch.set_default_dtype(torch.float32)
set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Optuna Search Configuration

In [2]:
import optuna

MLP_SEARCH_SPACE = {
    'hidden_layers': [1, 2, 3],
    'hidden_units': [15, 90, 104],
    'activation': ['Sine', 'Sigmoid', 'Tanh'],
    'learning_rate': [1e-4, 1e-3, 1e-2],
}


class Sine(nn.Module):
    def forward(self, x):
        return torch.sin(x)


ACTIVATIONS = {
    'Sine': lambda: Sine(),
    'Sigmoid': nn.Sigmoid,
    'Tanh': nn.Tanh,
}

N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
results_dir = f'results_mlp_semi_infinite_optuna_{timestamp}'
os.makedirs(results_dir, exist_ok=True)
print(f'Results will be saved to: {results_dir}')
print(f'Optuna trials: {N_TRIALS}')

Results will be saved to: results_mlp_semi_infinite_optuna_2026-09-20_07-42-39
Optuna trials: 50


## Objective Function

In [3]:
def objective(trial):
    """Run one semi-infinite MLP configuration and return mean global error."""
    config = {
        name: trial.suggest_categorical(name, values)
        for name, values in MLP_SEARCH_SPACE.items()
    }
    activation = ACTIVATIONS[config['activation']]()

    print(
        f"\n--- Trial {trial.number}: "
        f"L={config['hidden_layers']}, "
        f"N={config['hidden_units']}, "
        f"activation={config['activation']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        result = run_experiment_semi_inf(
            model_type='MLP',
            hidden_layers=config['hidden_layers'],
            hidden_units=config['hidden_units'],
            activation=activation,
            adam_lr=config['learning_rate'],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
            eval_domain=(-10.0, 10.0, -10.0, 0.0),
        )
    except Exception as error:
        print(f'Trial {trial.number} failed: {error}')
        raise optuna.exceptions.TrialPruned() from error

    err_u = float(result['err_u_global'])
    err_k = float(result['err_k_global'])
    compute_time = float(result['compute_time_sec'])
    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr('err_u', err_u)
    trial.set_user_attr('err_k', err_k)
    trial.set_user_attr('compute_time_sec', compute_time)

    print(
        f'Success! Time: {compute_time:.2f}s | '
        f'Err U: {err_u:.3e} | Err K: {err_k:.3e} | '
        f'Mean error: {mean_global_error:.3e}'
    )
    return mean_global_error

## Run Optimization

In [4]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name=f'mlp_semi_infinite_domain_{timestamp}',
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print('\n========================================')
print('BEST SEMI-INFINITE MLP CONFIGURATION')
print('========================================')
print(f'Mean global error: {study.best_value:.6e}')
print('Parameters:')
for name, value in study.best_params.items():
    print(f'  {name}: {value}')

[I 2026-09-20 07:43:06,110] A new study created in memory with name: mlp_semi_infinite_domain_2026-09-20_07-42-39



--- Trial 0: L=2, N=15, activation=Tanh, lr=1e-02 ---


/home/orincon/miniconda3/envs/PIKAN-unbounded-domains-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[I 2026-09-20 07:44:20,800] Trial 0 finished with value: 0.009054536844301611 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'activation': 'Tanh', 'learning_rate': 0.01}. Best is trial 0 with value: 0.009054536844301611.


Success! Time: 74.69s | Err U: 8.935e-03 | Err K: 9.174e-03 | Mean error: 9.055e-03

--- Trial 1: L=2, N=15, activation=Tanh, lr=1e-04 ---


[I 2026-09-20 07:45:41,772] Trial 1 finished with value: 0.02957282232726977 and parameters: {'hidden_layers': 2, 'hidden_units': 15, 'activation': 'Tanh', 'learning_rate': 0.0001}. Best is trial 0 with value: 0.009054536844301611.


...

Success! Time: 84.67s | Err U: 6.507e-03 | Err K: 7.741e-03 | Mean error: 7.124e-03

--- Trial 48: L=1, N=104, activation=Tanh, lr=1e-03 ---


[I 2026-09-20 08:48:11,431] Trial 48 finished with value: 0.16038644810908836 and parameters: {'hidden_layers': 1, 'hidden_units': 104, 'activation': 'Tanh', 'learning_rate': 0.001}. Best is trial 11 with value: 0.001966010234905468.


Success! Time: 77.45s | Err U: 2.524e-02 | Err K: 2.955e-01 | Mean error: 1.604e-01

--- Trial 49: L=1, N=104, activation=Sigmoid, lr=1e-02 ---


[I 2026-09-20 08:49:31,122] Trial 49 finished with value: 0.5343042044416965 and parameters: {'hidden_layers': 1, 'hidden_units': 104, 'activation': 'Sigmoid', 'learning_rate': 0.01}. Best is trial 11 with value: 0.001966010234905468.


Success! Time: 79.69s | Err U: 2.992e-02 | Err K: 1.039e+00 | Mean error: 5.343e-01

BEST SEMI-INFINITE MLP CONFIGURATION
Mean global error: 1.966010e-03
Parameters:
  hidden_layers: 2
  hidden_units: 104
  activation: Sigmoid
  learning_rate: 0.001


## Save Optimization Results

In [6]:
data_dir = os.path.join(results_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
joblib.dump(study, os.path.join(data_dir, 'study.pkl'))
joblib.dump(study, os.path.join(data_dir, f'study_{timestamp}.pkl'))
study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, 'study.csv')
study_df.to_csv(study_csv_path, index=False)
completed_df = study_df[
    study_df['state'].eq('COMPLETE')
].sort_values(by='value', ascending=True)
completed_csv_path = os.path.join(data_dir, 'study_completed_sorted.csv')
completed_df.to_csv(completed_csv_path, index=False)
print(f'Saved study to: {data_dir}')
print(f'Saved trial summary to: {study_csv_path}')
print(f'Saved sorted completed trials to: {completed_csv_path}')

Saved study to: results_mlp_semi_infinite_optuna_2026-09-20_07-42-39/data
Saved trial summary to: results_mlp_semi_infinite_optuna_2026-09-20_07-42-39/data/study.csv
Saved sorted completed trials to: results_mlp_semi_infinite_optuna_2026-09-20_07-42-39/data/study_completed_sorted.csv
